In [ ]:
import pandas as pd
import os


data_path = "D:\\Datastream\\PriceData\\US\\processed"

penny_percentile = 0.15
std_extreme_removal_threshold = 10


# read statics and determine delisting date
statics = pd.read_csv(os.path.join(data_path, "statics.csv"))
delist_str = statics["ENAME"].str.extract(r"DELIST\.(\d{2}/\d{2}/\d{2})")[0]
statics["DelistingDate"] = pd.to_datetime(delist_str, format="%d/%m/%y", errors="coerce")
statics['BDATE'] = pd.to_datetime(statics['BDATE'])

string_columns = ['DSCD', 'ENAME', 'EXMNEM', 'GEOGN', 'ISIN', 'ISINID', 'LOC', 'PCUR', 'TRAC', 'WC05601', 'TYPE', 'TR1N','TR2N','TR3N', 'CURRENCY']

if 'Type' in statics.columns:
    string_columns.insert(0, 'Type')

existing_string_columns = [col for col in string_columns if col in statics.columns]
statics[existing_string_columns] = statics[existing_string_columns].astype(str)
statics.loc[:, 'DSCD']      = statics['DSCD'].str.strip()

statics

,Type,DSCD,BDATE,ENAME,EXMNEM,GEOGN,ISIN,ISINID,LOC,PCUR,...,WC05601,RIC,TR1,TR1N,TR2,TR2N,TR3,TR3N,CURRENCY,DelistingDate
0,69568X,69568X,2010-06-29,TESLA,NAS,UNITED STATES,US88160R1014,P,U88160R101,U$,...,TSLA,TSLA.O,53.0,Consumer Cyclicals,5310.0,Automobiles & Auto Parts,531010.0,Automobiles & Auto Parts,U$,NaT
1,902242,902242,1973-01-02,JP MORGAN CHASE & COMPANY,NYS,UNITED STATES,US46625H1005,P,U46625H100,U$,...,JPM,JPM,55.0,Financials,5510.0,Banking & Investment Services,551010.0,Banking Services,U$,NaT
2,923937,923937,1973-01-02,BANK OF AMERICA,NYS,UNITED STATES,US0605051046,P,U060505104,U$,...,BAC,BAC,55.0,Financials,5510.0,Banking & Investment Services,551010.0,Banking Services,U$,NaT
3,904282,904282,1973-01-02,COCA COLA,NYS,UNITED STATES,US1912161007,P,U191216100,U$,...,KO,KO,54.0,Consumer Non-Cyclicals,5410.0,Food & Beverages,541010.0,Beverages,U$,NaT
4,906150,906150,1973-01-02,GE AEROSPACE,NYS,UNITED STATES,US3696043013,P,U369604301,U$,...,GE,GE,52.0,Industrials,5210.0,Industrial Goods,521010.0,Aerospace & Defense,U$,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38883,740646,740646,1985-02-11,LAIDLAW 'B' NON VOTING (NYS) DEAD - REORG IN 5...,NYS,UNITED STATES,CA50730K2065,S,U50730K206,U$,...,LDM,NaN,NaN,nan,NaN,nan,NaN,nan,U$,NaT
38884,27338Z,27338Z,2022-07-15,AMTD DIGITAL UNLISTED,NYS,UNITED STATES,nan,S,nan,U$,...,nan,NaN,NaN,nan,NaN,nan,NaN,nan,U$,NaT
38885,2608JK,2608JK,2021-10-06,LAND NEWCO INCOME DUMMY,NYS,UNITED STATES,nan,S,nan,U$,...,nan,NaN,NaN,nan,NaN,nan,NaN,nan,U$,NaT
38886,26532Z,26532Z,2021-10-22,PEGASUS DIGITAL MOBILITY ACQUISITION CL B,NYS,UNITED STATES,nan,S,nan,U$,...,SHMD,NaN,NaN,nan,NaN,nan,NaN,nan,U$,NaT


In [ ]:
price_panel = pd.concat([
    pd.read_feather(os.path.join(data_path, f"panel_{i:02d}.feather"))
    for i in range(1, 40)
]).reset_index(drop=True)

price_panel.rename(columns = {"DSCD": "Stock"}, inplace=True)

In [ ]:
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)


price_panel.sort_values(by=["Date", "Stock"], inplace=True)
price_panel.reset_index(drop=True, inplace=True)

logging.info(f"Number of rows before removing duplicate Stock-Date observations: {price_panel.shape[0]}")
price_panel = price_panel.drop_duplicates(subset=["Stock", "Date"], keep="first")
logging.info(f"Number of rows after removing duplicate Stock-Date observations: {price_panel.shape[0]}")

price_panel.loc[:, "Stock"] = price_panel["Stock"].astype(str)
price_panel.loc[:, "Stock"] = price_panel["Stock"].str.strip()

2026-04-09 19:50:35,338 : INFO : Number of rows before removing duplicate Stock-Date observations: 211846432
2026-04-09 19:52:16,337 : INFO : Number of rows after removing duplicate Stock-Date observations: 211846432


In [32]:
from datastream.preprocessing.filter import DSPreprocess
import numpy as np

In [33]:


mask = price_panel["ReturnIndex"] < 1e-6
logging.info(f"Frequency of ReturnIndex observations with extreme small values: {mask.sum()/price_panel.shape[0]:.6f}")

price_panel.loc[mask, "ReturnIndex"] = np.nan

price_panel["Return"] = (
    price_panel.groupby("Stock")["ReturnIndex"]
    .transform(lambda x: x / x.shift(1) - 1)
)

logging.info(f"Number of companies before removing non-regional companies: {statics.shape[0]}")
statics = statics[statics['GEOGN'] == 'UNITED STATES']
logging.info(f"Number of companies after removing non-regional companies: {statics.shape[0]}")

logging.info(f"Number of stocks with price data: {len(price_panel["Stock"].unique())}")

### Remove companies without return index information
logging.info(f"Observations of panel dataframe: {price_panel.shape[0]}")
# Filter (11) - Equity filter:
price_panel =  DSPreprocess.filter_companies_wo_return_index_data(price_panel)



2026-04-09 20:01:45,234 : INFO : Frequency of ReturnIndex observations with extreme small values: 0.000289
2026-04-09 20:03:11,379 : INFO : Number of companies before removing non-regional companies: 38888
2026-04-09 20:03:11,559 : INFO : Number of companies after removing non-regional companies: 38888
2026-04-09 20:03:19,982 : INFO : Number of stocks with price data: 38447
2026-04-09 20:03:19,983 : INFO : Observations of OHLCV panel dataframe: 211846432


MemoryError: Unable to allocate 16.4 GiB for an array with shape (11, 200458458) and data type float64

In [ ]:
########################################################################################################################
## Filters based on static data
########################################################################################################################
logging.info(f"Observations of panel dataframe: {price_panel.shape[0]}")
# Filter (1) - Equity filter:
price_panel = DSPreprocess.filter_non_common_stocks(price_panel, statics, country='UNITED STATES')

logging.info(f"Observations of panel dataframe: {price_panel.shape[0]}")
# Filter (2) - Cross-listing filter:
price_panel = DSPreprocess.filter_cross_listings(price_panel, statics, country='UNITED STATES')

logging.info(f"Observations of panel dataframe: {price_panel.shape[0]}")
# Filter (3): Duplicate LOC Codes
price_panel = DSPreprocess.filter_duplicate_loc_codes(price_panel, statics)

logging.info(f"Observations of panel dataframe: {price_panel.shape[0]}")
# Filter (4) - Foreign firms:
price_panel = price_panel[price_panel.Stock.isin(statics.DSCD.unique())]

logging.info(f"Observations of panel dataframe: {price_panel.shape[0]}")
# Filter (5) - Stocks in foreign currencies:
price_panel = DSPreprocess.filter_foreign_currency_stocks(price_panel, statics, country='UNITED STATES')

In [ ]:
########################################################################################################################
## Filters based on ReturnIndex
########################################################################################################################
logging.info(f"Observations of panel dataframe: {price_panel.shape[0]}")
# Filter (7) - :
# Remove stocks of which more than 98% of non-zero mean returns are either positive or negative
price_panel = DSPreprocess.filter_implausible_returns(price_panel)


Type                     object
DSCD                     object
BDATE            datetime64[ns]
ENAME                    object
EXMNEM                   object
GEOGN                    object
ISIN                     object
ISINID                   object
LOC                      object
PCUR                     object
TRAC                     object
TYPE                     object
WC05601                  object
RIC                      object
TR1                     float64
TR1N                     object
TR2                     float64
TR2N                     object
TR3                     float64
TR3N                     object
CURRENCY                 object
DelistingDate    datetime64[ns]
dtype: object

In [ ]:
########################################################################################################################
## Stockday filters:
########################################################################################################################
logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (13):
# If RI is forward filled for 10 consecutive days, then remove those days.
price_panel = DSPreprocess.filter_padded_values_delistings(price_panel, statics)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (12):
# If less than 120 observations.
price_panel = DSPreprocess.filter_short_history_stocks(price_panel, threshold=120)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (8):
# Remove stocks for which the returns are zero in more than 95% of their sample (After applying filter (13).
price_panel = DSPreprocess.filter_zero_return_stocks(price_panel)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (14):
# Stale prices
price_panel = DSPreprocess.filter_stale_prices(price_panel)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (9):
# Remove stocks with a daily standard deviation of more than 40%.
price_panel  = DSPreprocess.filter_stocks_by_high_volatility(price_panel, volatility_threshold=0.40)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (10):
# Remove stocks with a daily standard deviation of less than 0.01 bps.
price_panel = DSPreprocess.filter_stocks_by_low_volatility(price_panel)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (15):
# Target filter rate not reported / ~0.0015% (~0.00569% when applied on raw panel) actual filter rate
price_panel = DSPreprocess.filter_outlier_errors(price_panel, up_ts=1.0, down_ts=-0.5, method='drop')

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (16):
# Holiday filter: Has to be applied after filter (11) and (13)!
# Remove days on which non-missing or non-zero returns account for less than 0.5% of total available stocks.
price_panel = DSPreprocess.filter_holidays(price_panel)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (18):
# Filter Adjustment Inconsistencies
# Remove days with inconsistent adjustment values
price_panel = DSPreprocess.filter_adjustment_inconsistencies(price_panel, threshold = 0.05)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (Own - implausible OHLC):
# Nonsense values (Low > (Open OR High OR Close) and High < (Open OR Low OR Close):
price_panel = DSPreprocess.filter_implausible_prices(price_panel)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (No trading activity - Chaieb et al. (2021) JoFE)
price_panel = DSPreprocess.filter_no_trading_activity(price_panel)

logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
# Filter (Own - NA filter) - Drop all rows before they are populated for the first time and apply forward + backward fill.
price_panel = DSPreprocess.handle_missings(price_panel, statics, country='UNITED STATES')

# Filter (21) - Penny stocks.
logging.info(f"Observations of OHLCV panel dataframe: {price_panel.shape[0]}")
price_panel = DSPreprocess.filter_penny_stocks(price_panel, threshold=penny_percentile)
# logging.info("Saving monthly thresholds for penny stock selection.")